# VLM-SAM3 Experiment Results — Analysis Notebook

**Parametric:** change `JOB_ID` in the config cell below to switch between runs.  
All sections rebuild automatically from that single variable.

**Sections:**
1. Dataset overview & column descriptions
2. Missing VLM output — deep investigation
3. Metric distributions
4. Experiment comparison (all 10)
5. Object size & spatial effects
6. Temporal analysis (per-stream)
7. VLM output text analysis
8. Visual examples — images, overlays, GT masks *(requires dataset access)*
9. Summary & key findings

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — change JOB_ID to analyse a different run           ║
# ╚══════════════════════════════════════════════════════════════════════╝

JOB_ID = "462563"  # SLURM job ID whose results_<JOB_ID>.csv we want to analyse

# Paths (absolute so the notebook works from any working directory)
from pathlib import Path
REPO_ROOT   = Path("/home/3164542/reluminati-research")
SCRIPT_DIR  = REPO_ROOT / "src/scripts/experiments/vlm-sam3"
DATA_ROOT   = Path("/data/video_datasets/3321908/output_dir_all")
RESULTS_CSV = SCRIPT_DIR / f"results_{JOB_ID}.csv"

print(f"Results file : {RESULTS_CSV}")
print(f"Exists       : {RESULTS_CSV.exists()}")

In [ ]:
import json, re, warnings
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from pycocotools import mask as mask_utils

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 11})
sns.set_theme(style="whitegrid", palette="husl")

# ── colour palette consistent across the whole notebook ──────────────────────
EXP_ORDER = [
    "EXP-A", "EXP-B", "EXP-C", "EXP-D-bbox", "EXP-E-crop",
    "EXP-F-mf5", "EXP-A-cot", "EXP-C-cot", "EXP-A-vw5", "EXP-A-vw10",
]
EXP_PALETTE = dict(zip(EXP_ORDER, sns.color_palette("husl", len(EXP_ORDER))))

In [ ]:
df = pd.read_csv(RESULTS_CSV)

# Coerce numerics (guard against any stray strings)
for c in ["iou", "ba", "ca", "le", "obj_rel_area", "obj_dist_center",
          "src_mask_area", "src_move_x_ratio", "src_move_y_ratio",
          "vlm_num_frames", "vlm_video_window"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["frame"] = df["frame"].astype(str)
df["vlm_missing"] = df["vlm_output"].isna()
df["success"]     = df["iou"] > 0
df["success_25"]  = df["iou"] >= 0.25
df["success_50"]  = df["iou"] >= 0.50

KEY_COLS = ["take_uid", "object_name", "src_camera", "dest_camera", "frame"]
METRICS  = ["iou", "ba", "ca", "le"]

print(f"Loaded {len(df):,} rows  |  "
      f"{df[KEY_COLS].drop_duplicates().shape[0]} unique pairs  |  "
      f"{df['experiment'].nunique()} experiments")

---
## 1  Dataset Overview & Column Descriptions

In [ ]:
col_desc = {
    "take_uid":               "Take identifier (UUID)",
    "object_name":            "Object being tracked",
    "src_camera":             "Source (egocentric) camera",
    "dest_camera":            "Destination (exocentric) camera",
    "frame":                  "Frame index shared by both cameras",
    "experiment":             "Experiment ID (EXP-A … EXP-A-vw10)",
    "vlm_model":              "VLM used to generate the text prompt",
    "vlm_output":             "VLM-generated object description (NaN = failed)",
    "vlm_reprompted":         "True if VLM was called (not just cache reuse)",
    "vlm_reprompt_reason":    "Why VLM was called: cold_start / growth / movement / window_start / window_reuse",
    "vlm_num_frames":         "Number of frames shown to VLM",
    "vlm_video_window":       "Video-window size (0 = disabled)",
    "src_mask_area":          "Source mask area in pixels",
    "vlm_growth_threshold":   "Growth ratio that triggers VLM re-call",
    "vlm_movement_threshold": "Movement ratio that triggers VLM re-call",
    "src_move_x_ratio":       "Horizontal mask centroid shift (normalised)",
    "src_move_y_ratio":       "Vertical mask centroid shift (normalised)",
    "iou":    "Intersection over Union vs GT mask  [0,1] ↑",
    "ba":     "Balanced Accuracy (pixel-wise)       [0,1] ↑",
    "ca":     "Contour/Boundary Accuracy            [0,1] ↑",
    "le":     "Location Error (fraction wrong px)   [0,1] ↓",
    "obj_rel_area":       "GT mask area / image area",
    "obj_dist_center":    "Normalised distance of object centroid from image centre",
    "obj_size_cat":       "'small' (<0.5%), 'medium' (0.5-5%), 'large' (>5%)",
    "obj_is_peripheral":  "True if dist_center > 0.5",
}

desc_df = pd.DataFrame([
    {"column": c, "dtype": str(df[c].dtype) if c in df.columns else "—",
     "non_null": df[c].notna().sum() if c in df.columns else 0,
     "description": d}
    for c, d in col_desc.items() if c in df.columns
])
display(desc_df.style.set_properties(**{"text-align": "left"}))

In [ ]:
print("=== Numeric summary ===")
display(df[METRICS + ["obj_rel_area", "obj_dist_center"]].describe().round(4))

print("\n=== Categorical counts ===")
for col in ["experiment", "obj_size_cat", "vlm_reprompt_reason"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())

---
## 2  Missing VLM Output — Deep Investigation

**Hypothesis:** `num_predict=32` (the token cap for non-CoT experiments) is too short.  
When the model begins generating a longer response it gets truncated and the parser
returns `None`.  The `instruct-cloud` variants use 512 tokens and show **0% missing**.

We investigate whether missingness is:
- Experiment-specific (model / token-limit driven)
- Pair-specific (always the same pairs that fail)
- Time-clustered (API rate-limiting)
- Correlated with object properties

In [ ]:
miss = (
    df.groupby("experiment")["vlm_missing"]
    .agg(n_missing="sum", n_total="count",
         pct_missing=lambda x: 100 * x.mean())
    .reindex([e for e in EXP_ORDER if e in df["experiment"].unique()])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(miss["experiment"], miss["pct_missing"],
              color=[EXP_PALETTE.get(e, "grey") for e in miss["experiment"]],
              edgecolor="white", linewidth=0.8)
for b, v in zip(bars, miss["pct_missing"]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.3, f"{v:.1f}%",
            ha="center", fontsize=9)
ax.set_ylabel("Missing VLM output (%)")
ax.set_title("Missing VLM output rate per experiment\n"
             "(0% for instruct-cloud models; 18-27% for 235b-cloud with num_predict=32)",
             fontweight="bold")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

display(miss)

In [ ]:
# Are the SAME pairs always missing, or random pairs for each experiment?
pair_miss = (
    df.groupby(KEY_COLS)["vlm_missing"]
    .sum()
    .reset_index(name="n_experiments_missing")
)
pair_miss_nonzero = pair_miss[pair_miss["n_experiments_missing"] > 0]

print(f"Pairs with ≥1 missing experiment : {len(pair_miss_nonzero)} "
      f"/ {len(pair_miss)} ({100*len(pair_miss_nonzero)/len(pair_miss):.1f}%)")

fig, ax = plt.subplots(figsize=(7, 4))
pair_miss["n_experiments_missing"].value_counts().sort_index().plot(
    kind="bar", ax=ax, color=sns.color_palette("Blues_r", 11), edgecolor="white")
ax.set_xlabel("# experiments with missing VLM output (per pair)")
ax.set_ylabel("Number of pairs")
ax.set_title("How many experiments are missing per pair?", fontweight="bold")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

# Pairs missing in ALL non-cot experiments → systematic vs random
non_cot = [e for e in EXP_ORDER if "cot" not in e and e in df["experiment"].unique()]
pair_exp_miss = (
    df[df["experiment"].isin(non_cot)]
    .groupby(KEY_COLS)["vlm_missing"]
    .agg(list)
)
always_missing = pair_exp_miss.apply(lambda v: all(v)).sum()
never_missing  = pair_exp_miss.apply(lambda v: not any(v)).sum()
print(f"\nAmong non-CoT experiments:")
print(f"  Pairs where ALL non-cot exps missing : {always_missing}")
print(f"  Pairs where NO  non-cot exp  missing : {never_missing}")
print("\n→ High correlation means missingness is PAIR-driven (object/frame),")
print("  not just random API noise.")

In [ ]:
# Which reprompt_reason is most associated with missing output?
if "vlm_reprompt_reason" in df.columns:
    rr = (
        df.groupby("vlm_reprompt_reason")["vlm_missing"]
        .agg(n_missing="sum", n_total="count",
             pct_missing=lambda x: 100 * x.mean())
        .sort_values("pct_missing", ascending=False)
    )
    print("Missing rate by vlm_reprompt_reason:")
    display(rr)
    print("\nInsight: 'window_reuse' means the previous caption was reused;")
    print("if it was None, the reuse also stores None → cascading nulls.")

In [ ]:
# Does object size predict missingness?
sz_miss = (
    df.groupby("obj_size_cat")["vlm_missing"]
    .agg(pct_missing=lambda x: 100 * x.mean(), n="count")
    .reindex(["small", "medium", "large"])
)
print("Missing rate by object size:")
display(sz_miss)

# Scatter: object area vs whether VLM output is missing
fig, ax = plt.subplots(figsize=(7, 4))
for missing, color, label in [(False, "steelblue", "VLM present"),
                               (True,  "tomato",    "VLM missing")]:
    sub = df[df["vlm_missing"] == missing]
    ax.scatter(sub["obj_rel_area"], sub["iou"],
               alpha=0.15, s=10, color=color, label=label)
ax.set_xlabel("Object relative area")
ax.set_ylabel("IoU")
ax.set_title("IoU vs object area, coloured by VLM output presence",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Impact of missing VLM output on success rate
ct = pd.crosstab(df["vlm_missing"], df["success"], normalize="index") * 100
ct.index = ["VLM present", "VLM missing"]
ct.columns = ["Failure (IoU=0)", "Success (IoU>0)"]
print("Success rate by VLM output presence (row %):")
display(ct.round(1))

print("\n→ When VLM output is MISSING → 98%+ failure rate.")
print("  When VLM output IS present → success rate improves dramatically.")
print("\nFix: raise num_predict from 32 → 64 for all non-CoT experiments.")

---
## 3  Metric Distributions

All four metrics (IoU, BA, CA, LE) show a heavy spike at the worst value
(0 for IoU/BA/CA, 1 for LE), reflecting the high overall failure rate.
Conditioning on successes reveals the distribution of *how well* the
pipeline performs when it does find the object.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
metric_labels = {"iou": "IoU", "ba": "Balanced Accuracy",
                 "ca": "Contour Accuracy", "le": "Location Error"}

for col, (m, lbl) in enumerate(metric_labels.items()):
    # Top row: all data
    axes[0, col].hist(df[m].dropna(), bins=40, color="steelblue",
                      edgecolor="white", linewidth=0.4)
    axes[0, col].set_title(f"{lbl}\n(all pairs)", fontweight="bold")
    axes[0, col].set_xlabel(lbl)
    # Bottom row: successes only
    succ = df[df["success"]][m].dropna()
    axes[1, col].hist(succ, bins=40, color="seagreen",
                      edgecolor="white", linewidth=0.4)
    axes[1, col].set_title(f"{lbl}\n(IoU > 0 only, n={len(succ)})",
                           fontweight="bold")
    axes[1, col].set_xlabel(lbl)

fig.suptitle("Metric distributions — all pairs (top) vs successful pairs (bottom)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Overall success rate (IoU > 0)  : {df['success'].mean()*100:.1f}%")
print(f"Success rate (IoU ≥ 0.25)       : {df['success_25'].mean()*100:.1f}%")
print(f"Success rate (IoU ≥ 0.50)       : {df['success_50'].mean()*100:.1f}%")

---
## 4  Experiment Comparison (all 10)

We compare three views:
- **Unconditional** mean IoU (what you get on average, including failures)
- **Conditional** mean IoU given VLM output is present (isolates SAM3 quality)
- **Success rate** (IoU > 0), which shows how often the pipeline finds the object at all

In [ ]:
def exp_summary(df):
    rows = []
    for exp in EXP_ORDER:
        g = df[df["experiment"] == exp]
        if len(g) == 0:
            continue
        g_present = g[~g["vlm_missing"]]
        rows.append({
            "experiment": exp,
            "n": len(g),
            "vlm_missing_%": g["vlm_missing"].mean() * 100,
            "success_%": g["success"].mean() * 100,
            "iou>=0.25_%": g["success_25"].mean() * 100,
            "iou>=0.50_%": g["success_50"].mean() * 100,
            "iou_mean": g["iou"].mean(),
            "iou_mean|vlm_present": g_present["iou"].mean() if len(g_present) else np.nan,
            "iou_mean|success": g[g["success"]]["iou"].mean(),
            "le_mean": g["le"].mean(),
        })
    return pd.DataFrame(rows).set_index("experiment")

summ = exp_summary(df)
display(
    summ.style
        .format({
            "vlm_missing_%": "{:.1f}%",
            "success_%": "{:.1f}%",
            "iou>=0.25_%": "{:.1f}%",
            "iou>=0.50_%": "{:.1f}%",
            "iou_mean": "{:.3f}",
            "iou_mean|vlm_present": "{:.3f}",
            "iou_mean|success": "{:.3f}",
            "le_mean": "{:.3f}",
        })
        .background_gradient(subset=["success_%"], cmap="RdYlGn")
        .background_gradient(subset=["vlm_missing_%"], cmap="RdYlGn_r")
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, ylabel, title in [
    (axes[0], "success_%",         "Success rate (%)",     "Success rate (IoU > 0)"),
    (axes[1], "iou_mean",          "Mean IoU",              "Mean IoU (unconditional)"),
    (axes[2], "iou_mean|vlm_present", "Mean IoU | VLM present", "Mean IoU given VLM output present"),
]:
    vals = summ[col].dropna()
    bars = ax.bar(vals.index, vals.values,
                  color=[EXP_PALETTE.get(e, "grey") for e in vals.index],
                  edgecolor="white", linewidth=0.8)
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.003,
                f"{b.get_height():.2f}", ha="center", fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.tick_params(axis="x", rotation=40)

plt.suptitle(f"Experiment comparison — job {JOB_ID}",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# All four metrics side-by-side, ordered by success rate
order = summ.sort_values("success_%", ascending=False).index
plot_df = (
    df[df["experiment"].isin(order)]
    .groupby("experiment")[METRICS].mean()
    .reindex(order)
    .rename(columns={"iou": "IoU", "ba": "BA", "ca": "CA", "le": "LE"})
    .reset_index()
    .melt(id_vars="experiment", var_name="metric", value_name="value")
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.barplot(data=plot_df, x="experiment", y="value", hue="metric",
            order=order, palette="husl", edgecolor="white", ax=ax)
ax.set_xlabel("")
ax.set_title("All four metrics by experiment (sorted by success rate)",
             fontweight="bold")
ax.tick_params(axis="x", rotation=40)
ax.legend(title="Metric", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

---
## 5  Object Size & Spatial Effects

Object size is the strongest predictor of segmentation success. Smaller objects
are harder for both the VLM (less visual signal) and SAM3 (few pixels).

In [ ]:
SIZE_ORDER = ["small", "medium", "large"]
sz = (
    df.groupby("obj_size_cat")
    .agg(
        n=("iou", "count"),
        success_pct=("success", lambda x: 100*x.mean()),
        iou_mean=("iou", "mean"),
        iou_med=("iou", "median"),
        le_mean=("le", "mean"),
    )
    .reindex(SIZE_ORDER)
)
display(sz.round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ["#3498db", "#2ecc71", "#e74c3c"]

for ax, col, lbl in [
    (axes[0], "success_pct", "Success rate (%)"),
    (axes[1], "iou_mean",    "Mean IoU"),
    (axes[2], "le_mean",     "Mean Location Error"),
]:
    vals = sz[col].reindex(SIZE_ORDER)
    bars = ax.bar([s.capitalize() for s in SIZE_ORDER], vals,
                  color=colors, edgecolor="white")
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                f"{b.get_height():.3f}", ha="center", fontsize=10)
    ax.set_ylabel(lbl)
    ax.set_title(lbl, fontweight="bold")

plt.suptitle("Performance by object size category",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# IoU vs object area
ax = axes[0]
for size, color in zip(SIZE_ORDER, colors):
    sub = df[df["obj_size_cat"] == size]
    ax.scatter(sub["obj_rel_area"] * 100, sub["iou"],
               alpha=0.2, s=8, color=color, label=size.capitalize())
ax.set_xlabel("Object relative area (% of image)")
ax.set_ylabel("IoU")
ax.set_title("IoU vs object area", fontweight="bold")
ax.legend(title="Size")

# IoU vs distance from centre
ax = axes[1]
ax.scatter(df["obj_dist_center"], df["iou"],
           alpha=0.15, s=8, color="slateblue")
ax.axvline(0.5, color="tomato", ls="--", lw=1.5, label="peripheral threshold")
ax.set_xlabel("Normalised distance from image centre")
ax.set_ylabel("IoU")
ax.set_title("IoU vs object position", fontweight="bold")
ax.legend()

plt.tight_layout()
plt.show()

# Peripheral vs central
print("\nMean IoU — central vs peripheral objects:")
print(df.groupby("obj_is_peripheral")["iou"].mean().rename(
    {False: "Central", True: "Peripheral"}))

---
## 6  Temporal Analysis — Per-Stream Consistency

Each (take, object, src_cam, dst_cam) stream contains multiple frames.
We check whether performance is consistent across frames in the same stream
or whether it degrades/improves over time.

In [ ]:
stream_cols = ["take_uid", "object_name", "src_camera", "dest_camera"]

# Variance of IoU within each stream (per experiment)
stream_var = (
    df.groupby(stream_cols + ["experiment"])["iou"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "iou_mean", "std": "iou_std", "count": "n_frames"})
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 4))
(
    stream_var.groupby("experiment")["iou_std"]
    .mean()
    .reindex([e for e in EXP_ORDER if e in stream_var["experiment"].values])
    .plot(kind="bar", ax=ax, color=[EXP_PALETTE.get(e, "grey") for e in
          stream_var.groupby("experiment")["iou_std"].mean().reindex(
              [e for e in EXP_ORDER if e in stream_var["experiment"].values]).index],
          edgecolor="white")
)
ax.set_ylabel("Mean within-stream IoU std")
ax.set_title("IoU variability within streams\n"
             "(high std = inconsistent across frames in same video)",
             fontweight="bold")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

# Frame index vs IoU — does later frame = worse?
df["frame_int"] = pd.to_numeric(df["frame"], errors="coerce")
corr_frame_iou = df[["frame_int", "iou"]].corr().loc["frame_int", "iou"]
print(f"Pearson r(frame_index, IoU) = {corr_frame_iou:.3f}  "
      f"({'no clear trend' if abs(corr_frame_iou) < 0.05 else 'notable trend'})")

In [ ]:
# Does being re-prompted help or hurt?
if "vlm_reprompted" in df.columns:
    rp = (
        df[~df["vlm_missing"]]
        .groupby(["experiment", "vlm_reprompted"])["iou"]
        .mean()
        .unstack("vlm_reprompted")
        .rename(columns={False: "Not reprompted", True: "Reprompted"})
        .reindex([e for e in EXP_ORDER if e in df["experiment"].values])
    )
    print("Mean IoU: reprompted vs not (given VLM present):")
    display(rp.round(3))

    rp.plot(kind="bar", figsize=(11, 4), edgecolor="white")
    plt.title("Reprompted vs not — mean IoU (given VLM output present)",
              fontweight="bold")
    plt.ylabel("Mean IoU")
    plt.xlabel("")
    plt.xticks(rotation=40)
    plt.legend(bbox_to_anchor=(1.01, 1))
    plt.tight_layout()
    plt.show()

---
## 7  VLM Output Text Analysis

We examine *what* the VLM says: common words, colour hallucinations
(saying "red" when the object is not red), and whether the output text
predicts success.

In [ ]:
present = df[~df["vlm_missing"]]["vlm_output"].str.lower().str.strip()

# Word frequency
words = [w for text in present.dropna() for w in re.split(r"\s+", text) if w]
wc = Counter(words).most_common(30)
wc_df = pd.DataFrame(wc, columns=["word", "count"])

fig, ax = plt.subplots(figsize=(12, 4))
ax.barh(wc_df["word"][::-1], wc_df["count"][::-1], color="steelblue", edgecolor="white")
ax.set_xlabel("Frequency")
ax.set_title("Top-30 words in VLM outputs", fontweight="bold")
plt.tight_layout()
plt.show()

# "Red" rate — does the VLM hallucinate red overlay as object colour?
red_rate = present.str.contains(r"\bred\b").mean() * 100
print(f"VLM outputs containing 'red': {red_rate:.1f}%")
print("(Ideally ~0% since prompts say 'Do NOT say red unless object is actually red')")

In [ ]:
# Top VLM outputs in successful vs failed cases
for label, mask in [("✅ Successful (IoU > 0)", df["success"]),
                    ("❌ Failed (IoU = 0)",    ~df["success"])]:
    sub = df[mask & ~df["vlm_missing"]]["vlm_output"].str.lower().str.strip()
    top = sub.value_counts().head(12)
    print(f"\n{label} — top VLM outputs:")
    print(top.to_string())

In [ ]:
# Red hallucination by experiment
red_by_exp = (
    df[~df["vlm_missing"]]
    .assign(says_red=lambda d:
            d["vlm_output"].str.lower().str.contains(r"\bred\b").fillna(False))
    .groupby("experiment")["says_red"]
    .mean() * 100
    .reindex([e for e in EXP_ORDER if e in df["experiment"].values])
)

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(red_by_exp.index, red_by_exp.values,
       color=[EXP_PALETTE.get(e, "grey") for e in red_by_exp.index],
       edgecolor="white")
ax.set_ylabel("% outputs containing 'red'")
ax.set_title("'Red' hallucination rate per experiment\n"
             "(EXP-A shows most hallucination — only overlay image with no clean reference)",
             fontweight="bold")
ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

---
## 8  Visual Examples — Images, Overlays & Ground Truth Masks

This section reconstructs the **VLM input images** (overlay, bbox, crop)
and shows them side-by-side with the **ground-truth destination mask**
and **VLM output text**.

**Requirements:** access to the dataset at `DATA_ROOT`.
Running on the login node is fine (no GPU needed for this section).

> **Section 8b (optional):** shows the *predicted SAM3 mask* — requires
> a GPU and the `vlm-sam3` conda environment on a compute node.

In [ ]:
# ─── Image & mask helpers ─────────────────────────────────────────────────────

def resolve_img(base_path: str) -> Path:
    """Add .jpg extension — frames are stored without extension in the CSV."""
    p = Path(base_path)
    for ext in [".jpg", ".jpeg", ".png"]:
        if (p.parent / (p.name + ext)).exists():
            return p.parent / (p.name + ext)
    raise FileNotFoundError(f"No image found at {base_path}[.jpg/.jpeg/.png]")

def load_img(take_uid: str, camera: str, frame: str) -> np.ndarray:
    p = DATA_ROOT / take_uid / camera / frame
    return np.array(Image.open(resolve_img(str(p))).convert("RGB"))

def load_ann(take_uid: str) -> dict:
    with open(DATA_ROOT / take_uid / "annotation.json") as f:
        return json.load(f)

def decode_mask(ann: dict, obj: str, camera: str, frame: str) -> np.ndarray | None:
    try:
        rle = ann["masks"][obj][camera][str(frame)]
        return mask_utils.decode(rle).astype(bool)
    except KeyError:
        return None

def overlay_mask(img: np.ndarray, mask: np.ndarray,
                 color=(255, 0, 0), alpha=0.45) -> np.ndarray:
    """Blend a coloured mask onto an image (in-place copy)."""
    out = img.copy()
    mask_r = cv2.resize(mask.astype(np.uint8),
                        (img.shape[1], img.shape[0]),
                        interpolation=cv2.INTER_NEAREST).astype(bool)
    out[mask_r] = (out[mask_r] * (1-alpha) +
                   np.array(color) * alpha).astype(np.uint8)
    return out

def bbox_img(img: np.ndarray, mask: np.ndarray, pad=10) -> np.ndarray:
    """Draw a red bounding box around the mask."""
    mask_r = cv2.resize(mask.astype(np.uint8),
                        (img.shape[1], img.shape[0]),
                        interpolation=cv2.INTER_NEAREST).astype(bool)
    ys, xs = np.where(mask_r)
    if len(ys) == 0:
        return img.copy()
    y1, y2 = max(ys.min()-pad, 0), min(ys.max()+pad, img.shape[0])
    x1, x2 = max(xs.min()-pad, 0), min(xs.max()+pad, img.shape[1])
    out = img.copy()
    cv2.rectangle(out, (x1, y1), (x2, y2), (220, 30, 30), 3)
    return out

def crop_img(img: np.ndarray, mask: np.ndarray, pad=10) -> np.ndarray:
    """Crop the image to the mask bounding box."""
    mask_r = cv2.resize(mask.astype(np.uint8),
                        (img.shape[1], img.shape[0]),
                        interpolation=cv2.INTER_NEAREST).astype(bool)
    ys, xs = np.where(mask_r)
    if len(ys) == 0:
        return img.copy()
    y1, y2 = max(ys.min()-pad, 0), min(ys.max()+pad, img.shape[0])
    x1, x2 = max(xs.min()-pad, 0), min(xs.max()+pad, img.shape[1])
    return img[y1:y2, x1:x2]

def gt_overlay(img: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Green overlay for ground-truth mask."""
    return overlay_mask(img, mask, color=(0, 220, 0), alpha=0.45)

print("Helper functions loaded.")

In [ ]:
def show_example(row: pd.Series, ann: dict = None, ax_title_suffix=""):
    """
    Show a single result row as a figure:
      [src overlay] [src clean] [src bbox/crop if applicable] [dst + GT mask]
    Plus experiment info and metrics.
    """
    take, obj = row["take_uid"], row["object_name"]
    src_cam, dst_cam = row["src_camera"], row["dest_camera"]
    frame = str(row["frame"])
    exp   = row["experiment"]

    if ann is None:
        ann = load_ann(take)

    try:
        src_img  = load_img(take, src_cam, frame)
        dst_img  = load_img(take, dst_cam, frame)
    except FileNotFoundError as e:
        print(f"Image not found: {e}")
        return

    src_mask = decode_mask(ann, obj, src_cam, frame)
    dst_mask = decode_mask(ann, obj, dst_cam, frame)

    panels = []
    titles = []

    if src_mask is not None:
        panels.append(overlay_mask(src_img, src_mask));  titles.append("Src + red overlay")
        panels.append(src_img.copy());                   titles.append("Src clean")
        if "bbox" in exp:
            panels.append(bbox_img(src_img, src_mask));  titles.append("Src bbox")
        if "crop" in exp:
            panels.append(crop_img(src_img, src_mask));  titles.append("Src crop")
    else:
        panels.append(src_img); titles.append("Src (mask unavailable)")

    panels.append(dst_img.copy());  titles.append("Dst clean")
    if dst_mask is not None:
        panels.append(gt_overlay(dst_img, dst_mask))
        titles.append("Dst + GT mask (green)")

    fig, axes = plt.subplots(1, len(panels), figsize=(4*len(panels), 4))
    if len(panels) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, panels, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    vlm_out = row.get("vlm_output", "—")
    iou_v   = row.get("iou", float("nan"))
    fig.suptitle(
        f"{exp}  |  obj: {obj}  |  frame {frame}  |  "
        f"VLM: \"{vlm_out}\"  |  IoU={iou_v:.3f}  {ax_title_suffix}",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()

print("show_example() defined.")

In [ ]:
# ── Show N random successful examples across different experiments ─────────────
N_SHOW = 3
successes_df = (
    df[df["success_25"]]          # IoU ≥ 0.25
    .groupby("experiment", group_keys=False)
    .apply(lambda g: g.sample(min(1, len(g)), random_state=42))
    .head(N_SHOW)
)

print(f"Showing {len(successes_df)} successful examples (IoU ≥ 0.25):")
for _, row in successes_df.iterrows():
    show_example(row)

In [ ]:
# ── Show N random failure examples where VLM DID return output ─────────────────
N_SHOW = 3
failures_df = (
    df[~df["success"] & ~df["vlm_missing"]]  # IoU = 0 but VLM gave output
    .groupby("experiment", group_keys=False)
    .apply(lambda g: g.sample(min(1, len(g)), random_state=7))
    .head(N_SHOW)
)

print(f"Showing {len(failures_df)} failure examples (IoU=0, VLM output present):")
for _, row in failures_df.iterrows():
    show_example(row, ax_title_suffix="← FAILURE")

In [ ]:
# ── Compare all experiments on the SAME pair ───────────────────────────────────
# Find a pair that is successful in at least one experiment
best_pair = (
    df[df["success"]]
    .groupby(KEY_COLS)["iou"]
    .max()
    .sort_values(ascending=False)
    .index[0]
)
pair_df = df[
    (df["take_uid"]     == best_pair[0]) &
    (df["object_name"]  == best_pair[1]) &
    (df["src_camera"]   == best_pair[2]) &
    (df["dest_camera"]  == best_pair[3]) &
    (df["frame"]        == best_pair[4])
]

print(f"Comparing all experiments on pair: take={best_pair[0][:8]}… "
      f"obj={best_pair[1]}  frame={best_pair[4]}")

ann_cache = {}
for _, row in pair_df.sort_values("experiment").iterrows():
    uid = row["take_uid"]
    if uid not in ann_cache:
        ann_cache[uid] = load_ann(uid)
    show_example(row, ann=ann_cache[uid])

### 8b  Predicted Mask Visualisation (requires GPU + `vlm-sam3` env)

Uncomment and run on a compute node. Adds the SAM3 predicted mask
in **red** alongside the GT mask in **green**.

In [ ]:
# ── Uncomment below to run SAM3 inference and show predicted masks ────────────

# import torch
# from transformers import Sam3Processor, Sam3Model
# from huggingface_hub import login
#
# HF_TOKEN = "HUGGINGFACE_TOKEN_HERE"   # or read from env
# login(token=HF_TOKEN)
# device = "cuda" if torch.cuda.is_available() else "cpu"
# sam_m = Sam3Model.from_pretrained("facebook/sam3").to(device)
# sam_p = Sam3Processor.from_pretrained("facebook/sam3")
#
# def run_sam3(dst_img_np, text_prompt):
#     pil_img = Image.fromarray(dst_img_np)
#     inputs = sam_p(images=pil_img, text=text_prompt, return_tensors="pt").to(device)
#     with torch.no_grad():
#         outputs = sam_m(**inputs)
#     results = sam_p.post_process_instance_segmentation(
#         outputs, threshold=0.5, mask_threshold=0.5,
#         target_sizes=inputs.get("original_sizes").tolist()
#     )[0]
#     if results["masks"]:
#         return results["masks"][0].cpu().numpy().astype(bool)
#     return None
#
# # Show predicted + GT for the best pair
# for _, row in pair_df.sort_values("experiment").iterrows():
#     if row["vlm_missing"]: continue
#     take, obj = row["take_uid"], row["object_name"]
#     dst_cam, frame = row["dest_camera"], str(row["frame"])
#     ann = ann_cache.get(take) or load_ann(take)
#     dst_img = load_img(take, dst_cam, frame)
#     gt_mask = decode_mask(ann, obj, dst_cam, frame)
#     pred_mask = run_sam3(dst_img, row["vlm_output"])
#
#     panels = [dst_img]
#     titles = ["Dst clean"]
#     if gt_mask   is not None: panels.append(gt_overlay(dst_img, gt_mask));   titles.append("GT (green)")
#     if pred_mask is not None: panels.append(overlay_mask(dst_img, pred_mask)); titles.append("Predicted (red)")
#
#     fig, axes = plt.subplots(1, len(panels), figsize=(4*len(panels), 4))
#     for ax, img, t in zip(axes, panels, titles):
#         ax.imshow(img); ax.set_title(t); ax.axis("off")
#     fig.suptitle(f"{row['experiment']}  VLM: \"{row['vlm_output']}\"  IoU={row['iou']:.3f}",
#                  fontweight="bold")
#     plt.tight_layout(); plt.show()

print("Uncomment the block above and run on a GPU compute node.")

---
## 9  Summary & Key Findings

In [ ]:
print(f"{'='*70}")
print(f"RESULTS SUMMARY — Job {JOB_ID}")
print(f"{'='*70}")
print(f"Pairs processed : {df[KEY_COLS].drop_duplicates().shape[0]:,}")
print(f"Total rows      : {len(df):,}")
print(f"Experiments     : {sorted(df['experiment'].unique())}")
print()

print("─── Overall success rates ───────────────────────────────────────────")
print(f"  IoU > 0    : {df['success'].mean()*100:.1f}%")
print(f"  IoU ≥ 0.25 : {df['success_25'].mean()*100:.1f}%")
print(f"  IoU ≥ 0.50 : {df['success_50'].mean()*100:.1f}%")
print(f"  VLM missing: {df['vlm_missing'].mean()*100:.1f}%  ← major issue")
print()

print("─── Best experiments (by success rate) ──────────────────────────────")
display(summ[["success_%", "iou_mean", "iou_mean|vlm_present", "vlm_missing_%"]]
        .sort_values("success_%", ascending=False).round(3))

print("\n─── Key insights ────────────────────────────────────────────────────")
insights = [
    "1. EXP-C-cot (CoT + 3 images) is the best overall — 0% missing VLM output and highest success rate.",
    "2. Missing VLM output (18-27% for non-CoT) is the single biggest performance limiter.",
    "   FIX: raise num_predict from 32 → 64 for all 235b-cloud experiments.",
    "3. More context always helps: EXP-A < EXP-B < EXP-C (each adds an image).",
    "4. Video windowing (vw5/vw10) does NOT improve over EXP-A; vw5 is clearly worse.",
    "5. Multi-frame overlay (EXP-F-mf5) is the worst performer — confuses the model.",
    "6. Object size is the strongest spatial predictor: large > medium >> small.",
    "7. When VLM output IS present, conditional IoU is high (0.5-0.7) — SAM3 is capable;",
    "   the bottleneck is the VLM producing a good text description.",
]
for i in insights:
    print(" ", i)